# Supply Chain Delay Analysis

### Data Cleaning using Python

This notebook cleans the raw supply chain data so it is ready for analysis

**Steps in this notebook:**
1. Setup & Imports
2. Load the Raw Data
3. Data Understanding
4. Data Quality Assessment
5. Data Cleaning — Dimension Tables
6. Data Cleaning — Fact Table
7. Post-Cleaning Validation
8. Feature Engineering
9. Export Cleaned Data


## Step 1: Setup & Imports

**Objective:** Prepare the environment with the tools needed to work with the data.

**Description:** Pandas is used to load, clean, and work with tabular data. 


In [1]:
import pandas as pd

## Step 2: Load the Raw Data

**Objective:** Bring all source files into memory for inspection.

**Description:** Loads four dimension tables (location, products, supplier, transportation) and one fact table (supply_chain.csv, the transaction-level data) into separate DataFrames.


In [2]:
# Dimension tables 
location_df = pd.read_csv(r'D:\DIVYA\Data Analytics\NoviTech 6 Month Training Projects\4. Power BI\data\raw\location.csv')
products_df = pd.read_csv(r'D:\DIVYA\Data Analytics\NoviTech 6 Month Training Projects\4. Power BI\data\raw\products.csv')
supplier_df = pd.read_csv(r'D:\DIVYA\Data Analytics\NoviTech 6 Month Training Projects\4. Power BI\data\raw\supplier.csv')
transportation_df = pd.read_csv(r'D:\DIVYA\Data Analytics\NoviTech 6 Month Training Projects\4. Power BI\data\raw\transportation.csv')

# Fact table (the actual supply chain transactions)
transactions_df = pd.read_csv(r'D:\DIVYA\Data Analytics\NoviTech 6 Month Training Projects\4. Power BI\data\raw\transactions.csv')

print("All files loaded successfully!\n")


All files loaded successfully!



## Step 3: Data Understanding

**Objective:** Get familiar with the data's structure and content before changing anything.

**Description:** Checks table shapes, previews the fact table, reviews column names/types (.info()) and summary statistics (.describe()), and previews all four dimension tables.


### 3.1 Check the size of the tables

In [3]:
print("Locations:      ", location_df.shape)
print("Products:       ", products_df.shape)
print("Supplier:       ", supplier_df.shape)
print("Transportation: ", transportation_df.shape)
print("Transactions:   ", transactions_df.shape)

Locations:       (15, 8)
Products:        (20, 8)
Supplier:        (15, 9)
Transportation:  (15, 9)
Transactions:    (10536, 44)


### 3.2 Preview the fact table

In [4]:
# Look at the first 5 rows of the main transaction table
transactions_df.head()


,Transaction_ID,Product_ID,Supplier_ID,Location_ID,Transport_ID,Order_ID,Order_Date,Quantity_Ordered,Quantity_Delivered,Unit_Cost,...,Warehouse_Capacity_Utilization,Order_Weekday,Is_Weekend_Order,Order_Month_Name,Fulfillment_Rate_Pct,Delay_Severity,Total_Internal_Processing_Days,Cost_Per_Unit_Delivered,High_Value_Order_Flag,Weather_Risk_Flag
0,TXN109950,PROD009,SUP007,LOC006,TRN014,ORD103526,07/07/2026,31,31.0,624.5,...,44.0,Tuesday,No,July,100.0,No Delay,11,597.75,No,Yes
1,TXN102725,PROD013,SUP008,LOC013,TRN001,ORD105799,08 Oct 2025,17,17.0,1297.02,...,NaN,Wednesday,No,October,100.0,No Delay,11,1280.89,No,No
2,TXN109547,PROD009,SUP015,LOC011,TRN004,ORD101820,16 May 2026,5,5.0,700.05,...,63.8,Saturday,Yes,May,100.0,No Delay,9,695.50,No,No
3,TXN109242,PROD013,SUP014,LOC002,TRN005,ORD100851,25 Jan 2026,26,9.0,1544.24,...,87.2,Sunday,Yes,January,34.6,Minor (1-3 days),14,4380.83,No,No
4,TXN108598,PROD006,SUP008,LOC015,TRN015,ORD103107,10.09.2025,29,29.0,27729.97,...,92.3,Wednesday,No,September,100.0,Minor (1-3 days),16,27165.21,Yes,Yes


### 3.3 Column names and data types

In [5]:
# Column names, data types, and how many non-missing values each column has
transactions_df.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10536 entries, 0 to 10535
Data columns (total 44 columns):
 #   Column                          Non-Null Count  Dtype  
---  ------                          --------------  -----  
 0   Transaction_ID                  10536 non-null  object 
 1   Product_ID                      10536 non-null  object 
 2   Supplier_ID                     10536 non-null  object 
 3   Location_ID                     10536 non-null  object 
 4   Transport_ID                    10536 non-null  object 
 5   Order_ID                        10536 non-null  object 
 6   Order_Date                      10536 non-null  object 
 7   Quantity_Ordered                10536 non-null  int64  
 8   Quantity_Delivered              10139 non-null  float64
 9   Unit_Cost                       10536 non-null  object 
 10  Order_Value                     10536 non-null  float64
 11  Promised_Delivery_Date          10536 non-null  object 
 12  Actual_Delivery_Date            

### 3.4 Summary statistics

In [6]:
# Basic statistics (min, max, average, etc.) for all the numeric columns
transactions_df.describe()


,Quantity_Ordered,Quantity_Delivered,Order_Value,Delivery_Days,Delay_Days,Procurement_Days,Supplier_Delay_Days,Production_Days,Warehouse_Days,Transportation_Days,Temperature_C,Rainfall_MM,Visibility_KM,Traffic_Delay_Minutes,Inventory_Level,Warehouse_Capacity_Utilization,Fulfillment_Rate_Pct,Total_Internal_Processing_Days,Cost_Per_Unit_Delivered
count,10536.000000,10139.000000,1.053600e+04,9981.000000,9981.000000,10536.000000,10220.000000,10536.000000,10536.000000,10536.000000,10010.000000,10010.000000,10010.000000,10536.000000,10326.000000,10326.000000,10139.000000,10536.000000,9976.000000
mean,28.355733,27.006904,1.020490e+05,13.592325,0.674582,4.771355,0.210959,2.646260,1.998007,3.528853,29.068581,11.351139,6.305105,32.601936,912.527600,68.126409,95.197229,12.944476,3711.939465
std,46.837781,45.132321,4.060000e+05,3.349516,1.466153,2.038365,0.441418,1.670783,0.814962,1.088893,6.056347,27.261836,2.448242,37.410541,611.256465,16.588041,17.098407,2.965400,6666.499402
min,-5.000000,0.000000,4.179500e+02,3.000000,-1.000000,1.000000,0.000000,0.000000,1.000000,2.000000,6.100000,0.000000,0.300000,0.000000,0.000000,40.000000,0.000000,4.000000,376.300000
25%,11.000000,10.000000,1.190956e+04,11.000000,0.000000,3.000000,0.000000,1.000000,1.000000,3.000000,24.900000,0.900000,4.800000,8.000000,364.000000,54.200000,100.000000,11.000000,732.972500
50%,21.000000,20.000000,2.453032e+04,13.000000,0.000000,5.000000,0.000000,2.000000,2.000000,3.000000,29.100000,1.900000,6.600000,20.000000,901.000000,67.900000,100.000000,13.000000,1417.100000
75%,31.000000,30.000000,5.994321e+04,16.000000,1.000000,6.000000,0.000000,4.000000,3.000000,4.000000,33.200000,2.800000,8.300000,43.000000,1438.000000,81.800000,100.000000,15.000000,2563.935000
max,395.000000,395.000000,1.034937e+07,27.000000,13.000000,12.000000,3.000000,9.000000,3.000000,6.000000,59.000000,179.900000,10.000000,179.000000,1999.000000,139.500000,100.000000,25.000000,66136.440000


### 3.5 Preview the dimension tables

In [7]:
print("LOCATION TABLE")
location_df.head()


LOCATION TABLE


,Location_ID,Warehouse_Name,Location_Type,City,State,Region,Warehouse_Capacity,Warehouse_Status
0,LOC001,BLR Fulfillment Center 1,Fulfillment Center,Bengaluru,Karnataka,South,500000,Active
1,LOC002,Mumbai Distribution Hub,Distribution Center,Mumbai,Maharashtra,West,350000,Active
2,LOC003,Delhi NCR Warehouse,Warehouse,Gurugram,Haryana,North,600000,Active
3,LOC004,Chennai Fulfillment Center,Fulfillment Center,Chennai,Tamil Nadu,South,420000,Active
4,LOC005,Hyderabad Warehouse,Warehouse,Hyderabad,Telangana,South,380000,Active


In [8]:
print("PRODUCTS TABLE")
products_df.head()

PRODUCTS TABLE


,Product_ID,Product_Name,Category,Sub_Category,Unit_of_Measure,Product_Priority,Standard_Lead_Time_Days,Product_Status
0,PROD001,Wireless Bluetooth Earbuds,Electronics,Audio,Unit,High,5,Active
1,PROD002,Smartphone 5G 128GB,Electronics,Mobiles & Accessories,Unit,High,7,Active
2,PROD003,Men's Cotton T-Shirt,Fashion,Men's Apparel,Unit,Medium,3,Active
3,PROD004,Women's Kurti Set,Fashion,Women's Apparel,Unit,Medium,4,Active
4,PROD005,Non-Stick Cookware Set,Home & Kitchen,Cookware,Set,Medium,6,Active


In [9]:
print("SUPPLIER TABLE")
supplier_df.head()

SUPPLIER TABLE


,Supplier_ID,Supplier_Name,Supplier_Type,Supplier_City,Supplier_State,Supplier_Region,Supplier_Rating,Standard_Lead_Time_Days,Supplier_Status
0,SUP001,Om Sai Electronics Pvt Ltd,Manufacturer,Chennai,Tamil Nadu,South,4.2,5,Active
1,SUP002,Shree Ganesh Textiles,Manufacturer,Surat,Gujarat,West,3.8,7,Active
2,SUP003,Bharat Home Appliances Ltd,Manufacturer,Noida,Uttar Pradesh,North,4.0,6,Active
3,SUP004,Nova Distributors,Distributor,Bengaluru,Karnataka,South,3.5,4,Active
4,SUP005,Global Mobile Traders,Distributor,Mumbai,Maharashtra,West,4.5,3,Active


In [10]:
print("TRANSPORTATION TABLE")
transportation_df.head()

TRANSPORTATION TABLE


,Transport_ID,Transport_Mode,Vehicle_Type,Carrier_Name,Route_ID,Origin_City,Destination_City,Distance_KM,Planned_Transit_Hours
0,TRN001,Road,Mini Truck,Delhivery,RT1001,Bengaluru,Chennai,350,8
1,TRN002,Road,Truck,Ekart Logistics,RT1002,Mumbai,Pune,150,4
2,TRN003,Air,Air Cargo,Blue Dart Aviation,RT1003,Delhi,Bengaluru,1740,3
3,TRN004,Road,Truck,XpressBees,RT1004,Delhi,Jaipur,280,6
4,TRN005,Rail,Freight Train,Indian Railways Freight,RT1005,Mumbai,Delhi,1400,24


## Step 4: Data Quality Assessment

**Objective:** Diagnose exactly what's wrong with the raw data before fixing it.

**Description:** Checks for missing values, duplicate rows, inconsistent text casing/spacing, a text-formatted Unit_Cost (currency symbol + commas), five mixed date formats, and invalid values (negative quantities, delivered > ordered, placeholder costs like -100 and 999999). Ends with a written issue summary.

### 4.1 Missing Value Analysis

In [11]:
# Count how many values are missing in each column
missing = transactions_df.isnull().sum()

missing = pd.DataFrame({
    'Column': missing.index,
    'Missing Values': missing.values
})

missing


,Column,Missing Values
0,Transaction_ID,0
1,Product_ID,0
2,Supplier_ID,0
3,Location_ID,0
4,Transport_ID,0
5,Order_ID,0
6,Order_Date,0
7,Quantity_Ordered,0
8,Quantity_Delivered,397
9,Unit_Cost,0


### 4.2 Duplicate Check

In [12]:
print("Duplicate rows:   ", transactions_df.duplicated().sum())
print("Duplicate Transaction_ID:", transactions_df['Transaction_ID'].duplicated().sum())


Duplicate rows:    0
Duplicate Transaction_ID: 36


### 4.3 Inconsistent text values

Identify unique values in a few text columns. Notice the extra spaces and the mix of `UPPERCASE` / `lowercase` / `Title Case` — these all mean the same thing but are stored differently.


In [13]:
print('CATEGORICAL COLUMNS')
print()
print('Delay_Status: ', transactions_df['Delay_Status'].unique())
print()
print('Weather_Condition: ', transactions_df['Weather_Condition'].unique())
print()
print('Traffic_Level: ', transactions_df['Traffic_Level'].unique())
print()
print('Road_Condition: ', transactions_df['Road_Condition'].unique())
print()
print('Demand_Level: ', transactions_df['Demand_Level'].unique())
print()

print('FLAGGED COLUMNS')
print()
print('Road_Closure_Flag: ', transactions_df['Road_Closure_Flag'].unique())
print()
print('Accident_Flag: ', transactions_df['Accident_Flag'].unique())
print()
print('Stockout_Flag: ', transactions_df['Stockout_Flag'].unique())
print()

CATEGORICAL COLUMNS

Delay_Status:  ['Early' 'On Time' 'Delayed' 'delayed' ' On Time ' 'Delayed  '
 'In Transit' 'cancelled' ' Early ' 'DELAYED' 'On Time  ' ' Delayed '
 'on time' 'EARLY' 'Early  ' 'early' 'ON TIME' 'Cancelled' ' In Transit '
 ' Cancelled ' 'IN TRANSIT' 'Cancelled  ' 'in transit' 'In Transit  '
 'CANCELLED']

Weather_Condition:  ['Stormy' 'CLOUDY' 'Clear' 'Extreme Heat' 'Rainy' nan 'Cloudy' 'Foggy'
 'clear' 'Stormy  ' 'Rainy  ' 'RAINY' 'Clear  ' 'CLEAR' ' Foggy ' 'foggy'
 'FOGGY' ' Cloudy ' 'Foggy  ' ' Clear ' 'rainy' ' Extreme Heat ' 'STORMY'
 'Cloudy  ' 'EXTREME HEAT' 'cloudy' 'extreme heat' ' Stormy ' ' Rainy '
 'Extreme Heat  ' 'stormy']

Traffic_Level:  ['Low' 'High' 'Medium' 'MEDIUM' 'Severe' 'Medium  ' ' Severe ' 'medium'
 'HIGH' ' High ' 'low' 'high' ' Medium ' 'Low  ' 'LOW' 'SEVERE' 'Severe  '
 ' Low ' 'severe' 'High  ']

Road_Condition:  ['Normal' 'Wet' 'Flooded' 'POOR' 'Poor' 'NORMAL' 'Normal  ' ' Wet '
 'normal' ' Normal ' 'Wet  ' 'WET' 'wet' 'poor' 'Poor  

### 4.4 Check Data Types

`Unit_Cost` should hold numbers, but pandas has loaded it as text.


In [14]:
# Unit_Cost should hold numbers, but pandas has loaded it as text.
print("Current data type of Unit_Cost:", transactions_df['Unit_Cost'].dtype)

# Show a few values that contain the "Rs." currency symbol or a comma
sample = transactions_df[transactions_df['Unit_Cost'].str.contains('Rs|,', na=False)]
sample['Unit_Cost'].head()


Current data type of Unit_Cost: object


10      Rs.573.59
21      Rs.482.32
29    Rs.3,607.48
33      Rs.480.45
36      Rs.461.26
Name: Unit_Cost, dtype: object

In [15]:
# Dates stored as text, in several formats
transactions_df['Order_Date'].sample(10, random_state=1)

3817    23 May 2026
2621     19.06.2026
6717     14-04-2026
3397     07.03.2026
5825     03.01.2026
6567     21.04.2026
9312     28-05-2026
1589     17-06-2026
5753    26 Apr 2026
1508     19-05-2026
Name: Order_Date, dtype: object

### 4.5 Invalid numeric values

In [16]:
# Quantity_Ordered should never be negative
print("Orders with a negative Quantity_Ordered:", (transactions_df['Quantity_Ordered'] < 0).sum())

# Quantity_Delivered should never be more than what was ordered
print("Orders where Quantity_Delivered > Quantity_Ordered:",
      (transactions_df['Quantity_Delivered'] > transactions_df['Quantity_Ordered']).sum())


Orders with a negative Quantity_Ordered: 30
Orders where Quantity_Delivered > Quantity_Ordered: 40


### 4.6 Suspicious placeholder values

Check the lowest and highest unit costs — real prices shouldn't include strange extremes.


In [17]:
# Turn Unit_Cost into numbers just for this check (we clean it properly in Step 6)
temp_unit_cost = transactions_df['Unit_Cost'].str.replace('Rs.', '', regex=False)
temp_unit_cost = temp_unit_cost.str.replace(',', '', regex=False).astype(float)

print("5 lowest unit costs: ", sorted(temp_unit_cost.unique())[:5])
print("5 highest unit costs:", sorted(temp_unit_cost.unique())[-5:])


5 lowest unit costs:  [np.float64(-100.0), np.float64(377.16), np.float64(382.31), np.float64(385.36), np.float64(386.08)]
5 highest unit costs: [np.float64(30103.04), np.float64(30502.5), np.float64(30651.44), np.float64(31410.13), np.float64(999999.0)]


### Summary of issues found

- **Missing values** in several columns
- **36 duplicate transactions** (same `Transaction_ID` appears twice)
- **Inconsistent text**: extra spaces and mixed CAPITALIZATION in status/weather/flag columns
- **Unit_Cost stored as text**, with a `Rs.` symbol and comma separators
- **Dates stored as text**, mixing 5 different formats (`YYYY-MM-DD`, `DD-MM-YYYY`, `DD.MM.YYYY`, `MM/DD/YYYY`, and `DD Mon YYYY`)
- **Placeholder/error values**: some unit costs are `-100` or `999999`, which aren't real prices
- **30 orders with a negative `Quantity_Ordered`** (not possible in real life)
- **40 orders where `Quantity_Delivered` is more than `Quantity_Ordered`** (not possible in real life)


## Step 5: Data Cleaning — Dimension Tables

**Objective:** Clean the reference tables first, since the fact table depends on them.

**Description:** Checks each dimension table for missing values/duplicates (finds none), strips stray whitespace as a defensive measure, drops any duplicate rows, and confirms each ID column is a unique key.

### 5.1 Check missing values and duplicate rows

In [18]:
tables = [
    ('location_df', location_df),
    ('products_df', products_df),
    ('supplier_df', supplier_df),
    ('transportation_df', transportation_df),
]

for name, df in tables:
    print(name, "-> missing values:", df.isnull().sum().sum(), "| duplicate rows:", df.duplicated().sum())


location_df -> missing values: 0 | duplicate rows: 0
products_df -> missing values: 0 | duplicate rows: 0
supplier_df -> missing values: 0 | duplicate rows: 0
transportation_df -> missing values: 0 | duplicate rows: 0


All four dimension tables already come back with **0 missing values and 0 duplicate rows**.


### 5.2 Remove extra spaces from text columns

In [19]:
# List the text columns in each table
location_text_cols = ['Warehouse_Name', 'Location_Type', 'City', 'State', 'Region', 'Warehouse_Status']
products_text_cols = ['Product_Name', 'Category', 'Sub_Category', 'Unit_of_Measure', 'Product_Priority', 'Product_Status']
supplier_text_cols = ['Supplier_Name', 'Supplier_Type', 'Supplier_City', 'Supplier_State', 'Supplier_Region', 'Supplier_Status']
transportation_text_cols = ['Transport_Mode', 'Vehicle_Type', 'Carrier_Name', 'Origin_City', 'Destination_City']

# Remove any extra spaces from the start/end of each text value
for col in location_text_cols:
    location_df[col] = location_df[col].str.strip()

for col in products_text_cols:
    products_df[col] = products_df[col].str.strip()

for col in supplier_text_cols:
    supplier_df[col] = supplier_df[col].str.strip()

for col in transportation_text_cols:
    transportation_df[col] = transportation_df[col].str.strip()

print("Whitespace cleaned.")


Whitespace cleaned.


### 5.3 Remove duplicate rows (defensive check)

In [20]:
location_df = location_df.drop_duplicates()
products_df = products_df.drop_duplicates()
supplier_df = supplier_df.drop_duplicates()
transportation_df = transportation_df.drop_duplicates()

print("Shapes after cleaning:")
print("location_df:      ", location_df.shape)
print("products_df:      ", products_df.shape)
print("supplier_df:      ", supplier_df.shape)
print("transportation_df:", transportation_df.shape)


Shapes after cleaning:
location_df:       (15, 8)
products_df:       (20, 8)
supplier_df:       (15, 9)
transportation_df: (15, 9)


### 5.4 Confirm the ID columns are unique

In [21]:
# Each ID column should act as a unique key for that table
print("Location_ID unique?  ", location_df['Location_ID'].is_unique)
print("Product_ID unique?   ", products_df['Product_ID'].is_unique)
print("Supplier_ID unique?  ", supplier_df['Supplier_ID'].is_unique)
print("Transport_ID unique? ", transportation_df['Transport_ID'].is_unique)


Location_ID unique?   True
Product_ID unique?    True
Supplier_ID unique?   True
Transport_ID unique?  True


## Step 6: Data Cleaning — Fact Table

**Objective:** Fix every issue identified in Step 4.

**Description:** Removes 36 duplicate transactions; standardizes categorical text to Title Case; maps every yes/no variant to a clean Yes/No; converts Unit_Cost to numeric and fixes placeholder values; removes negative-quantity orders and caps over-delivered quantities; parses the five date formats into real dates; fills missing values with column-specific rules (e.g., undelivered orders left blank on purpose); fixes data types; and drops the 6 columns with no dashboard value

### 6.1 Remove duplicate transactions

Each `Transaction_ID` should appear only once. Keep the first occurrence of each one and drop the rest.


In [22]:
print("Rows before removing duplicates:", len(transactions_df))

transactions_df = transactions_df.drop_duplicates(subset='Transaction_ID', keep='first')

print("Rows after removing duplicates: ", len(transactions_df))


Rows before removing duplicates: 10536
Rows after removing duplicates:  10500


### 6.2 Standardize Categorical Columns

Remove extra spaces and make the capitalization consistent (Title Case) for the descriptive text columns.


In [23]:
fact_text_cols = ['Delay_Status', 'Delay_Reason', 'Weather_Condition',
                   'Traffic_Level', 'Road_Condition', 'Demand_Level']

for col in fact_text_cols:
    transactions_df[col] = transactions_df[col].str.strip().str.title()
    print(f'{col}: ', transactions_df[col].unique())
    print()


Delay_Status:  ['Early' 'On Time' 'Delayed' 'In Transit' 'Cancelled']

Delay_Reason:  [nan 'Stockout' 'Poor Road Conditions' 'Supplier Delay'
 'Traffic Congestion' 'Weather Conditions' 'Road Closure'
 'Accident En Route' 'Order Cancelled']

Weather_Condition:  ['Stormy' 'Cloudy' 'Clear' 'Extreme Heat' 'Rainy' nan 'Foggy']

Traffic_Level:  ['Low' 'High' 'Medium' 'Severe']

Road_Condition:  ['Normal' 'Wet' 'Flooded' 'Poor']

Demand_Level:  ['High' 'Normal' 'Very High' 'Low']



### 6.3 Standardize Yes/No flag columns

`Road_Closure_Flag`, `Accident_Flag`, and `Stockout_Flag` all mean the same thing (yes or no) but are written in many different ways: `Yes`, `Y`, `True`, `1`, `No`, `N`, `False`, `0`. We map every version to a clean `Yes` / `No`.


In [24]:
# Every way "yes" and "no" show up in the raw data, mapped to a clean value
yes_no_map = {
    'yes': 'Yes', 'y': 'Yes', 'true': 'Yes', '1': 'Yes',
    'no': 'No', 'n': 'No', 'false': 'No', '0': 'No',
}

flag_cols = ['Road_Closure_Flag', 'Accident_Flag', 'Stockout_Flag']

for col in flag_cols:
    transactions_df[col] = transactions_df[col].str.strip().str.lower().map(yes_no_map)
    print(f'{col}: ', transactions_df[col].unique())


Road_Closure_Flag:  ['No' 'Yes']
Accident_Flag:  ['No' 'Yes']
Stockout_Flag:  ['No' 'Yes']


In [25]:
# A few other flag columns are already clean - just remove any stray spaces
transactions_df['High_Value_Order_Flag'] = transactions_df['High_Value_Order_Flag'].str.strip()
transactions_df['Weather_Risk_Flag'] = transactions_df['Weather_Risk_Flag'].str.strip()

print("Done.")


Done.


### 6.4 Convert Text to Numeric

`Unit_Cost` is stored as text because some values include a `Rs.` symbol and commas. A few values are also placeholder errors: `-100` (an impossible negative price) and `999999` (an unrealistically high price used as a stand-in for "unknown"). We'll convert the column to numbers and treat those two placeholder values as missing, then fill them in with the median (the typical middle value).


In [26]:
# Remove the "Rs." currency symbol
transactions_df['Unit_Cost'] = transactions_df['Unit_Cost'].str.replace('Rs.', '', regex=False)

# Remove commas used as thousand separators
transactions_df['Unit_Cost'] = transactions_df['Unit_Cost'].str.replace(',', '', regex=False)

# Convert the column from text to numbers
transactions_df['Unit_Cost'] = transactions_df['Unit_Cost'].astype(float)

# -100 and 999999 are placeholder/error values, not real prices - treat them as missing
transactions_df['Unit_Cost'] = transactions_df['Unit_Cost'].replace([-100, 999999], float('nan'))

# Fill the now-missing values with the median unit cost
transactions_df['Unit_Cost'] = transactions_df['Unit_Cost'].fillna(transactions_df['Unit_Cost'].median())

transactions_df['Unit_Cost'].describe()


count    10500.000000
mean      3479.031992
std       6076.921396
min        377.160000
25%        714.702500
50%       1367.840000
75%       2451.202500
max      31410.130000
Name: Unit_Cost, dtype: float64

### 6.5 Fix invalid quantities

A negative `Quantity_Ordered` isn't a real order, so those rows are removed. A `Quantity_Delivered` that is higher than `Quantity_Ordered` isn't possible either,  cap it at the ordered amount.


In [27]:
print("Rows before removing invalid orders:", len(transactions_df))

# Remove rows where the ordered quantity is negative
transactions_df = transactions_df[transactions_df['Quantity_Ordered'] >= 0]

print("Rows after removing invalid orders: ", len(transactions_df))


Rows before removing invalid orders: 10500
Rows after removing invalid orders:  10470


In [28]:
# Quantity_Delivered can never be more than Quantity_Ordered - cap it if this happens
transactions_df['Quantity_Delivered'] = transactions_df[['Quantity_Delivered', 'Quantity_Ordered']].min(axis=1)

# Check the fix worked
print("Orders where Quantity_Delivered > Quantity_Ordered:", (transactions_df['Quantity_Delivered'] > transactions_df['Quantity_Ordered']).sum())


Orders where Quantity_Delivered > Quantity_Ordered: 0


### 6.6 Fix the mixed date formats

- Dates written with **slashes** (like `09/25/2026`) follow the **Month/Day/Year** order.
- Dates written with **dashes**, **dots**, or a **month name** (like `25-09-2026`, `25.09.2026`, `25 Sep 2026`) follow the **Day/Month/Year** order.
- Dates already written as `2026-09-25` are unambiguous.

Parse the slash-style dates on their own using Month/Day/Year, parse everything else using Day/Month/Year, and combine the two results.


In [29]:
def clean_date_column(column):
    # True for rows where the original text contains a slash
    slash_dates = column.str.contains('/', na=False)

    # Parse assuming Month/Day/Year (for the slash-style dates)
    parsed_slash = pd.to_datetime(column, format='%m/%d/%Y', errors='coerce')

    # Parse assuming Day/Month/Year (for every other style)
    parsed_other = pd.to_datetime(column, format='mixed', dayfirst=True, errors='coerce')

    # Use the slash-style result where the original text had a slash,
    # otherwise use the other result
    return parsed_slash.where(slash_dates, parsed_other)


transactions_df['Order_Date'] = clean_date_column(transactions_df['Order_Date'])
transactions_df['Promised_Delivery_Date'] = clean_date_column(transactions_df['Promised_Delivery_Date'])
transactions_df['Actual_Delivery_Date'] = clean_date_column(transactions_df['Actual_Delivery_Date'])

transactions_df[['Order_Date', 'Promised_Delivery_Date', 'Actual_Delivery_Date']].head()


,Order_Date,Promised_Delivery_Date,Actual_Delivery_Date
0,2026-07-07,2026-07-18,2026-07-17
1,2025-10-08,2025-10-19,2025-10-19
2,2026-05-16,2026-05-25,2026-05-25
3,2026-01-25,2026-02-08,2026-02-10
4,2025-09-10,2025-09-26,2025-09-27


### 6.7 Handle missing values

Different columns are missing values for different reasons, treat each case on its own terms.

In [30]:
# Delay_Reason is blank when there was no delay
transactions_df['Delay_Reason'] = transactions_df['Delay_Reason'].fillna('No Delay')

# Weather_Condition is blank when it wasn't recorded
transactions_df['Weather_Condition'] = transactions_df['Weather_Condition'].fillna('Unknown')

# Orders still "In Transit" haven't delivered anything yet
transactions_df['Quantity_Delivered'] = transactions_df['Quantity_Delivered'].fillna(0)
transactions_df['Fulfillment_Rate_Pct'] = transactions_df['Fulfillment_Rate_Pct'].fillna(0)

# Fill weather/inventory numbers with the median (the typical middle value)
transactions_df['Temperature_C'] = transactions_df['Temperature_C'].fillna(transactions_df['Temperature_C'].median())
transactions_df['Rainfall_MM'] = transactions_df['Rainfall_MM'].fillna(transactions_df['Rainfall_MM'].median())
transactions_df['Visibility_KM'] = transactions_df['Visibility_KM'].fillna(transactions_df['Visibility_KM'].median())
transactions_df['Inventory_Level'] = transactions_df['Inventory_Level'].fillna(transactions_df['Inventory_Level'].median())
transactions_df['Warehouse_Capacity_Utilization'] = transactions_df['Warehouse_Capacity_Utilization'].fillna(
    transactions_df['Warehouse_Capacity_Utilization'].median()
)

# No recorded supplier delay means 0 days
transactions_df['Supplier_Delay_Days'] = transactions_df['Supplier_Delay_Days'].fillna(0)

print("Missing values remaining:")
transactions_df.isnull().sum()


Missing values remaining:


Transaction_ID                      0
Product_ID                          0
Supplier_ID                         0
Location_ID                         0
Transport_ID                        0
Order_ID                            0
Order_Date                          0
Quantity_Ordered                    0
Quantity_Delivered                  0
Unit_Cost                           0
Order_Value                         0
Promised_Delivery_Date              0
Actual_Delivery_Date              551
Delivery_Days                     551
Delay_Days                        551
Delay_Status                        0
Delay_Reason                        0
Procurement_Days                    0
Supplier_Delay_Days                 0
Production_Days                     0
Warehouse_Days                      0
Transportation_Days                 0
Weather_Condition                   0
Temperature_C                       0
Rainfall_MM                         0
Visibility_KM                       0
Traffic_Leve

`Actual_Delivery_Date`, `Delivery_Days`, `Delay_Days`, and `Cost_Per_Unit_Delivered` still show missing values — and that's expected. Those rows belong to orders that are still **In Transit** or were **Cancelled**, so there is no delivery date, delay, or delivered cost yet. Filling those in with a made-up number would be misleading, so leave them blank on purpose.

### 6.8 Tidy up data types

In [31]:
# Now that these columns have no missing values left, they can be whole numbers
transactions_df['Quantity_Delivered'] = transactions_df['Quantity_Delivered'].astype(int)
transactions_df['Supplier_Delay_Days'] = transactions_df['Supplier_Delay_Days'].astype(int)

print("Data types updated.")


Data types updated.


### 6.9 Drop unnecessary columns

In [32]:
cols_to_drop = [
    'Order_Weekday', 'Is_Weekend_Order', 'Order_Month_Name',
    'Total_Internal_Processing_Days', 'Cost_Per_Unit_Delivered',
]

In [33]:
transactions_df = transactions_df.drop(columns=cols_to_drop)

## Step 7: Post-Cleaning Validation

**Objective:** Confirm every Step 4 issue is actually resolved.

**Description:** Re-runs the Step 4 checks against the cleaned data, adds basic business-rule checks (delivered > ordered, delivery before order date, negative cost/value), and previews the result.

### 7.1 Missing values

In [34]:
# Missing values should now only remain where they are expected
# (undelivered orders don't have delivery info yet)
transactions_df.isnull().sum()


Transaction_ID                      0
Product_ID                          0
Supplier_ID                         0
Location_ID                         0
Transport_ID                        0
Order_ID                            0
Order_Date                          0
Quantity_Ordered                    0
Quantity_Delivered                  0
Unit_Cost                           0
Order_Value                         0
Promised_Delivery_Date              0
Actual_Delivery_Date              551
Delivery_Days                     551
Delay_Days                        551
Delay_Status                        0
Delay_Reason                        0
Procurement_Days                    0
Supplier_Delay_Days                 0
Production_Days                     0
Warehouse_Days                      0
Transportation_Days                 0
Weather_Condition                   0
Temperature_C                       0
Rainfall_MM                         0
Visibility_KM                       0
Traffic_Leve

### 7.2 Duplicate transactions

In [35]:
print("Duplicate Transaction_ID:", transactions_df['Transaction_ID'].duplicated().sum())


Duplicate Transaction_ID: 0


### 7.3 Text columns are now consistent

In [36]:
print("Delay_Status:       ", transactions_df['Delay_Status'].unique())
print("Weather_Condition:  ", transactions_df['Weather_Condition'].unique())
print("Road_Closure_Flag:  ", transactions_df['Road_Closure_Flag'].unique())


Delay_Status:        ['Early' 'On Time' 'Delayed' 'In Transit' 'Cancelled']
Weather_Condition:   ['Stormy' 'Cloudy' 'Clear' 'Extreme Heat' 'Rainy' 'Unknown' 'Foggy']
Road_Closure_Flag:   ['No' 'Yes']


### 7.4 Unit_Cost is numeric and realistic

In [37]:
print("Data type:      ", transactions_df['Unit_Cost'].dtype)
print("Minimum value:  ", transactions_df['Unit_Cost'].min())
print("Maximum value:  ", transactions_df['Unit_Cost'].max())


Data type:       float64
Minimum value:   377.16
Maximum value:   31410.13


### 7.5 Quantities are valid

In [38]:
print("Negative Quantity_Ordered:", (transactions_df['Quantity_Ordered'] < 0).sum())
print("Quantity_Delivered > Quantity_Ordered:",
      (transactions_df['Quantity_Delivered'] > transactions_df['Quantity_Ordered']).sum())


Negative Quantity_Ordered: 0
Quantity_Delivered > Quantity_Ordered: 0


### 7.6 Dates are now real date values

In [39]:
transactions_df[['Order_Date', 'Promised_Delivery_Date', 'Actual_Delivery_Date']].dtypes

Order_Date                datetime64[ns]
Promised_Delivery_Date    datetime64[ns]
Actual_Delivery_Date      datetime64[ns]
dtype: object

### 7.7 Check Basic Business Rules
**Description:** These checks help identify values that do not make business sense.

In [40]:
print('Delivered quantity greater than ordered quantity:', (transactions_df['Quantity_Delivered'] > transactions_df['Quantity_Ordered']).sum())
print('Actual delivery before order date:', (transactions_df['Actual_Delivery_Date'] < transactions_df['Order_Date']).sum())
print('Negative Unit Cost:', (transactions_df['Unit_Cost'] < 0).sum())
print('Negative Order Value:', (transactions_df['Order_Value'] < 0).sum())


Delivered quantity greater than ordered quantity: 0
Actual delivery before order date: 0
Negative Unit Cost: 0
Negative Order Value: 0


## Step 8: Feature Engineering columns

**Objective:** Add the calculated fields needed to answer the problem statement's core questions — where delays originate and what they cost.

**Description:** Derives Primary_Bottleneck_Stage and Bottleneck_Days (root-cause identification across the 4 stages); Promised_Lead_Time_Days (order-to-promised-delivery gap); Order_Fulfillment_Gap and Undelivered_Order_Value (business/financial impact of shortfalls); and Delay_Percentage_of_Promised (delay normalized against promised time).

### 8.1 Root-Cause / Bottleneck Identification

In [41]:
stage_cols = ['Procurement_Days', 'Production_Days', 'Warehouse_Days', 'Transportation_Days']

# Which stage took the most days, for each order?
transactions_df['Primary_Bottleneck_Stage'] = transactions_df[stage_cols].idxmax(axis=1)

# How many days did that slowest stage actually take?
transactions_df['Bottleneck_Days'] = transactions_df[stage_cols].max(axis=1)

transactions_df['Primary_Bottleneck_Stage'].value_counts()

Primary_Bottleneck_Stage
Procurement_Days       7364
Transportation_Days    1569
Production_Days        1391
Warehouse_Days          146
Name: count, dtype: int64

### 8.2 Time-based Features

In [42]:
# How many days did we promise, from order date to promised delivery date?
transactions_df['Promised_Lead_Time_Days'] = (
    transactions_df['Promised_Delivery_Date'] - transactions_df['Order_Date']
).dt.days

print(transactions_df['Promised_Lead_Time_Days'].describe())

count    10470.000000
mean        12.942789
std          2.968081
min          4.000000
25%         11.000000
50%         13.000000
75%         15.000000
max         25.000000
Name: Promised_Lead_Time_Days, dtype: float64


### 8.3 Business Impact & Customer Satisfaction

In [43]:
# How many units short of the order were actually delivered?
transactions_df['Order_Fulfillment_Gap'] = (
    transactions_df['Quantity_Ordered'] - transactions_df['Quantity_Delivered']
)

# What is that shortfall worth, in money? (using the per-unit cost)
transactions_df['Undelivered_Order_Value'] = (
    transactions_df['Order_Fulfillment_Gap'] * transactions_df['Unit_Cost']
)

print(transactions_df['Order_Fulfillment_Gap'].describe())
print()
print(transactions_df['Undelivered_Order_Value'].describe())


count    10470.000000
mean         1.369150
std         10.317216
min          0.000000
25%          0.000000
50%          0.000000
75%          0.000000
max        345.000000
Name: Order_Fulfillment_Gap, dtype: float64

count    1.047000e+04
mean     4.363351e+03
std      8.394825e+04
min      0.000000e+00
25%      0.000000e+00
50%      0.000000e+00
75%      0.000000e+00
max      7.343743e+06
Name: Undelivered_Order_Value, dtype: float64


### 8.4 Delay % of Promised Lead Time

In [44]:
# Delay as a percentage of how much time was promised
transactions_df['Delay_Percentage_of_Promised'] = (
    transactions_df['Delay_Days'] / transactions_df['Promised_Lead_Time_Days'] * 100
)

print(transactions_df['Delay_Percentage_of_Promised'].describe())


count    9919.000000
mean        5.446697
std        12.294247
min       -25.000000
25%         0.000000
50%         0.000000
75%        11.111111
max       118.181818
Name: Delay_Percentage_of_Promised, dtype: float64


### 8.5 Final Overview

All checks pass. The data is now clean, consistent, and ready to export.

In [45]:
print("Final shape of the cleaned fact table:", transactions_df.shape)

Final shape of the cleaned fact table: (10470, 45)


In [46]:
transactions_df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 10470 entries, 0 to 10535
Data columns (total 45 columns):
 #   Column                          Non-Null Count  Dtype         
---  ------                          --------------  -----         
 0   Transaction_ID                  10470 non-null  object        
 1   Product_ID                      10470 non-null  object        
 2   Supplier_ID                     10470 non-null  object        
 3   Location_ID                     10470 non-null  object        
 4   Transport_ID                    10470 non-null  object        
 5   Order_ID                        10470 non-null  object        
 6   Order_Date                      10470 non-null  datetime64[ns]
 7   Quantity_Ordered                10470 non-null  int64         
 8   Quantity_Delivered              10470 non-null  int64         
 9   Unit_Cost                       10470 non-null  float64       
 10  Order_Value                     10470 non-null  float64       
 11  Promise

### 8.6 Final Preview of the cleaned Fact table

In [47]:
transactions_df.head()

,Transaction_ID,Product_ID,Supplier_ID,Location_ID,Transport_ID,Order_ID,Order_Date,Quantity_Ordered,Quantity_Delivered,Unit_Cost,...,Fulfillment_Rate_Pct,Delay_Severity,High_Value_Order_Flag,Weather_Risk_Flag,Primary_Bottleneck_Stage,Bottleneck_Days,Promised_Lead_Time_Days,Order_Fulfillment_Gap,Undelivered_Order_Value,Delay_Percentage_of_Promised
0,TXN109950,PROD009,SUP007,LOC006,TRN014,ORD103526,2026-07-07,31,31,624.50,...,100.0,No Delay,No,Yes,Procurement_Days,7,11,0,0.00,-9.090909
1,TXN102725,PROD013,SUP008,LOC013,TRN001,ORD105799,2025-10-08,17,17,1297.02,...,100.0,No Delay,No,No,Procurement_Days,5,11,0,0.00,0.000000
2,TXN109547,PROD009,SUP015,LOC011,TRN004,ORD101820,2026-05-16,5,5,700.05,...,100.0,No Delay,No,No,Procurement_Days,3,9,0,0.00,0.000000
3,TXN109242,PROD013,SUP014,LOC002,TRN005,ORD100851,2026-01-25,26,9,1544.24,...,34.6,Minor (1-3 days),No,No,Transportation_Days,6,14,17,26252.08,14.285714
4,TXN108598,PROD006,SUP008,LOC015,TRN015,ORD103107,2025-09-10,29,29,27729.97,...,100.0,Minor (1-3 days),Yes,Yes,Production_Days,6,16,0,0.00,6.250000


## Step 9: Export Cleaned Data

**Objective:** Save the analysis-ready tables for Power BI.

**Description:** Writes all five cleaned DataFrames to new CSV files in a processed folder, leaving the original raw files untouched.

In [48]:
output_file = 'D:/DIVYA/Data Analytics/NoviTech 6 Month Training Projects/4. Power BI/data/processed/supply_chain_cleaned.xlsx'

In [49]:
with pd.ExcelWriter(output_file, engine='openpyxl') as writer:
    location_df.to_excel(writer, sheet_name='Location', index=False)
    products_df.to_excel(writer, sheet_name='Products', index=False)
    supplier_df.to_excel(writer, sheet_name='Supplier', index=False)
    transportation_df.to_excel(writer, sheet_name='Transportation', index=False)
    transactions_df.to_excel(writer, sheet_name='Transactions', index=False)

print('All cleaned tables have been exported successfully to a single Excel workbook.')
print(f'Workbook: {output_file}')
print('Worksheets created:')
print(' - Location')
print(' - Products')
print(' - Supplier')
print(' - Transportation')
print(' - Transactions')

All cleaned tables have been exported successfully to a single Excel workbook.
Workbook: D:/DIVYA/Data Analytics/NoviTech 6 Month Training Projects/4. Power BI/data/processed/supply_chain_cleaned.xlsx
Worksheets created:
 - Location
 - Products
 - Supplier
 - Transportation
 - Transactions


### Done!

The data is now clean and consolidated, addressing the fragmented, error-prone raw data described in the project's problem statement. From here, the cleaned CSV files can be loaded directly into Power BI to build the Supply Chain Delay Analysis Dashboard.
